# 01 · Exploring the sources of an ETL pipeline

**Demo «Python and VS Code in ETL pipelines»**

Before building a pipeline, we need to understand the data it will process. In this notebook, we open the source files, examine them and look for the problems they contain. This is the same work that is done using the data preview features of a graphical tool: here, we do it by writing a few lines of code and immediately looking at the result.

**What we will do, step by step**

1. Prepare the working environment.
2. Look at which files are available.
3. Read the transactions file as it is.
4. Hunt for problems in the data.
5. Develop a transformation: first «as it is», then in the idiomatic way.
6. See how the same logic is used in the project, in the form of reusable functions.
7. Query the destination database after running the pipeline.

## Assumptions

Let us first list what we assume to be true in this notebook. If any of these assumptions does not hold, some cells may not work or may produce results that differ from those described.

1. **Environment.** The notebook is run in VS Code using the `etl_demo` conda environment, created as described in the `README.md` file. The *kernel* selector in the top right must show `etl_demo`.
2. **Location.** The notebook is in the `notebook` folder of the `demo_etl_python` project and has not been moved elsewhere.
3. **Data.** The `dati/input` folder contains the files produced by the `strumenti/genera_dati.py` script. These are synthetic, that is, invented data: no customer, account or transaction is real.
4. **File format.** The CSV files use a semicolon as the column separator and UTF-8 encoding. In the transactions file, dates are written in day/month/year format (for example, `17/11/2026`) and amounts in Italian format, with a period for thousands and a comma for decimals (for example, `1.234,56`).
5. **Reference day.** We work with the transactions file for 17 November 2026.
6. **pandas version.** The notebook was tested with pandas 3.0.
7. **Last section.** Section 7 requires the pipeline to have been run at least once for 17 November 2026: the pipeline creates the database we will query.

## The terms we will use

| Term | Meaning |
|---|---|
| **ETL** | Stands for *Extract, Transform, Load*: extract data from the sources, transform it and load it into the destination. |
| **Source** | The place where the data comes from. Here: CSV files and an Excel file. |
| **Destination** | The place where the transformed data is written. Here: a SQLite database. |
| **CSV** | A text file in which each line is a record and columns are separated by a character, in our case a semicolon. |
| **Notebook** | A document made up of *cells*: some contain text (like this one), others contain code that we can run one cell at a time, immediately seeing the result below the cell. |
| **Kernel** | The Python program that runs the notebook cells. Choosing the kernel means choosing the conda environment to use. |
| **pandas** | The most widely used Python library for working with tables of data. |
| **DataFrame** | A pandas table: rows and columns, like a spreadsheet or a database table. |
| **Rejected record** | A row that fails a check and is therefore not loaded. We do not throw it away: we set it aside together with the reason. |
| **Processing date** | The day to which the data we are loading refers. |

## 1 · Preparing the working environment

Let us import the libraries and check which Python we are working with.

In [ ]:
# Import the tools we need.
import sys                  # information about the Python interpreter in use
import time                 # to measure how long an operation takes
from pathlib import Path    # to specify folders and files in the same way on Windows, Linux and macOS

import pandas as pd         # the library for data tables ("pd" is the abbreviation everyone uses)

print("Versione di Python:", sys.version.split()[0])
print("Versione di pandas:", pd.__version__)
print("Interprete in uso: ", sys.executable)

**What to check.** The last printed line shows the path to the *interpreter*, that is, the `python` program running the cells. The path must contain the name of the conda environment, `etl_demo`. If it does not, we are using a different Python from the one prepared for the demo: it is best to stop and choose the correct kernel using the selector in the top right.

Now we locate the project's main folder and read the parameters file. We use the same function as the actual pipeline: this way, the notebook and the pipeline read files from the same folders.

In [ ]:
# Locate the project root folder.
# A notebook opened in VS Code normally runs in the folder that contains it (here: notebook).
# In that case, the project folder is the folder immediately above it.
RADICE_PROGETTO = Path.cwd()
if not (RADICE_PROGETTO / "config.toml").exists():
    RADICE_PROGETTO = RADICE_PROGETTO.parent

# Add the project folder to the list of locations where Python looks for modules:
# this allows us to import our "etl" package from here too.
if str(RADICE_PROGETTO) not in sys.path:
    sys.path.insert(0, str(RADICE_PROGETTO))

from etl.configurazione import carica_configurazione

config = carica_configurazione()                        # reads config.toml
CARTELLA_INPUT = config["percorsi"]["cartella_input"]   # the folder containing the source files

print("Cartella del progetto:     ", RADICE_PROGETTO)
print("Cartella dei file di input:", CARTELLA_INPUT)

## 2 · Which files are available

Let us list the contents of the input folder, with the size of each file.

In [ ]:
# Iterate over the files in the input folder in alphabetical order
for file in sorted(CARTELLA_INPUT.iterdir()):
    dimensione_kb = file.stat().st_size / 1024          # the size is in bytes: convert it to kilobytes
    print(f"{file.name:<26} {dimensione_kb:>10.1f} KB")

| File | Contents | Role in the pipeline |
|---|---|---|
| `movimenti_AAAAMMGG.csv` | One day's current account transactions. One file arrives each day. | This is the main source: the file to check and load. |
| `conti.csv` | Account master data: the customer, branch, type and status for each account. | Lookup table: adds the customer and branch to the transactions. |
| `clienti.csv` | Customer master data, with the commercial segment. | Lookup table: adds the segment to the transactions. |
| `cambi.xlsx` | Daily exchange rates against the euro, in an Excel file. | Used to convert transactions in other currencies into euros. |

## 3 · Reading the transactions file as it is

Let us read the file for 17 November 2026, asking pandas to **interpret nothing**: all columns are read as plain text. We will soon see why this caution matters.

In [ ]:
FILE_MOVIMENTI = CARTELLA_INPUT / "movimenti_20261117.csv"

grezzi = pd.read_csv(
    FILE_MOVIMENTI,
    sep=";",                  # the column separator is a semicolon
    dtype=str,                # read ALL columns as text
    keep_default_na=False,    # an empty cell remains empty text "", rather than becoming a "missing value"
    encoding="utf-8",
)

print("Righe lette:", len(grezzi), "- Colonne:", len(grezzi.columns))
grezzi.head(10)               # head(10) displays the first ten rows

**In VS Code.** The **Open 'grezzi' in Data Wrangler** button appears below the table. Data Wrangler is a VS Code extension that opens the table in an interactive grid: we can sort and filter the columns and, for each one, see how many distinct values and missing values it contains. It is the closest tool to the data preview in a graphical product. The same tool can also be opened without writing code: right-click a CSV file in the file list and then choose **Open in Data Wrangler**.

### Why read everything as text

We could ask pandas to convert the amounts immediately, telling it that the comma separates decimals and the period separates thousands. This seems convenient, but let us look at what happens with a file of just two rows, in the second of which someone has mistakenly used a period as the decimal separator.

In [ ]:
import io   # allows us to treat text as if it were a file

# A two-row "file": in row A, the amount is written in Italian format (1.234,56),
# in row B, it is written with a decimal point (1234.56). The two amounts are equal.
file_di_esempio = io.StringIO("id;importo\nA;1.234,56\nB;1234.56\n")

lettura_automatica = pd.read_csv(file_di_esempio, sep=";", decimal=",", thousands=".")
lettura_automatica

The amount in row B has become **123456,00**: a hundred times the correct value, without any warning. pandas did what we asked it to do: it treated every period as a thousands separator and removed it.

This leads to the rule we follow throughout the pipeline:

> **Check first, then convert.** We read the data as text, verify that each value matches the expected format and convert only the values that pass the check. The others become rejected records.

## 4 · Hunting for problems

Let us examine the file with five checks. For each one, we count the rows involved and look at a few examples.

### 4.1 · Invalid dates

Let us try converting the dates column from text to the «date» type. We specify the expected format with `%d/%m/%Y`, which means day, month and four-digit year separated by slashes.

In [ ]:
# With errors="coerce", dates that do not match the format do not cause an error:
# they become NaT ("Not a Time", meaning "missing date"), and we can count them.
date_convertite = pd.to_datetime(grezzi["data_operazione"], format="%d/%m/%Y", errors="coerce")

data_non_valida = date_convertite.isna()    # a True/False column: True where the date is invalid

print("Righe con la data non valida:", data_non_valida.sum())

# value_counts counts how often each value appears: see which dates are incorrect
grezzi.loc[data_non_valida, "data_operazione"].value_counts()

We find impossible dates (31 February, the date made up entirely of zeros), empty dates and dates written in a format different from the agreed one.

### 4.2 · Invalid amounts

For amounts, we describe the expected format with a **regular expression**, that is, a pattern that specifies what a piece of text must look like. Ours reads as follows:

| Part of the pattern | Meaning |
|---|---|
| `-?` | an optional minus sign |
| `\d{1,3}(\.\d{3})+` | one to three digits, followed by one or more groups consisting of a period and three digits (the thousands) |
| `\|\d+` | or, alternatively, a simple sequence of digits without periods |
| `,\d{2}` | a comma followed by exactly two decimal digits |

The `str.fullmatch` method checks that **all** the text, from beginning to end, matches the pattern.

In [ ]:
FORMATO_IMPORTO = r"-?(\d{1,3}(\.\d{3})+|\d+),\d{2}"

importo_valido = grezzi["importo"].str.fullmatch(FORMATO_IMPORTO)   # True where the amount matches the format

# The ~ symbol means "not": ~importo_valido is True where the amount is NOT valid
print("Righe con l'importo non valido:", (~importo_valido).sum())
grezzi.loc[~importo_valido, "importo"].value_counts()

Among the rejected amounts is `1234.56`, the case in the previous example: this time, we catch it instead of turning it into an incorrect value.

### 4.3 · Codes written in different ways

Let us count how many times each value appears in the `divisa` column.

In [ ]:
grezzi["divisa"].value_counts()

The same currency is written in several ways: `EUR`, `eur`, ` eur` with a leading space. To a person, they are the same thing; to a program, they are three different values. This is not a reason for rejection: we only need to **clean up** the code by removing spaces on either side (`strip`) and converting everything to uppercase (`upper`).

In [ ]:
divisa_pulita = grezzi["divisa"].str.strip().str.upper()
divisa_pulita.value_counts()

There are now five currencies. Notice the presence of `JPY`, the Japanese yen: we will return to it in section 4.6.

### 4.4 · Duplicate rows

Each transaction has an identifier, `id_movimento`, which must be unique. Let us check whether any identifier appears more than once.

In [ ]:
ripetuto = grezzi["id_movimento"].duplicated()   # True from the second occurrence of the same identifier onwards
print("Righe ripetute:", ripetuto.sum())

# Look at a concrete example: take the first repeated identifier and display all its rows
esempio = grezzi.loc[ripetuto, "id_movimento"].iloc[0]
grezzi[grezzi["id_movimento"] == esempio]

The two rows are identical: the same transaction was sent twice. In the pipeline, we will keep the first and reject the duplicate; otherwise, that transaction would be counted twice.

### 4.5 · Accounts that do not exist in the master data

Let us read the account master data and check that the account for each transaction is present. Before comparing, we clean up the account code, as we did for the currency.

In [ ]:
conti = pd.read_csv(CARTELLA_INPUT / "conti.csv", sep=";", dtype=str)
print("Conti in anagrafica:", len(conti))

conto_pulito = grezzi["id_conto"].str.strip().str.upper()
conto_presente = conto_pulito.isin(conti["id_conto"])     # True if the transaction account exists in the account master data

print("Movimenti su conti che non esistono in anagrafica:", (~conto_presente).sum())
grezzi[~conto_presente].head()

A transaction on an unknown account cannot be assigned to any customer or branch: it will be rejected.

### 4.6 · Currencies with missing exchange rates

Let us read the exchange rate table from the Excel file and look at the rates for 17 November. The exchange rate indicates how many units of the currency are needed to buy 1 euro.

In [ ]:
cambi = pd.read_excel(CARTELLA_INPUT / "cambi.xlsx", sheet_name="cambi")
cambi[cambi["data"] == "2026-11-17"]

In [ ]:
# Compare two sets: the currencies used in transactions and the currencies for which we have an exchange rate.
# Subtracting sets returns the elements of the first set that are not in the second.
divise_dei_movimenti = set(divisa_pulita)
divise_con_cambio = set(cambi["divisa"])
divise_senza_cambio = divise_dei_movimenti - divise_con_cambio

print("Divise usate nei movimenti ma assenti dalla tabella dei cambi:", divise_senza_cambio)
print("Movimenti coinvolti:", divisa_pulita.isin(divise_senza_cambio).sum())

Transactions in yen cannot be converted to euros because the exchange rate table does not contain the yen: they will be rejected.

### Summary of the problems found

Let's collect what we have discovered and the decision made for each problem in a table.

In [ ]:
riepilogo_problemi = pd.DataFrame(
    {
        "problema": [
            "Data non valida",
            "Importo non valido",
            "Riga ripetuta",
            "Conto inesistente",
            "Cambio mancante",
            "Codice scritto in modo diverso",
        ],
        "righe": [
            int(data_non_valida.sum()),
            int((~importo_valido).sum()),
            int(ripetuto.sum()),
            int((~conto_presente).sum()),
            int(divisa_pulita.isin(divise_senza_cambio).sum()),
            int((grezzi["divisa"] != divisa_pulita).sum()),
        ],
        "decisione": ["scartare", "scartare", "scartare", "scartare", "scartare", "ripulire e tenere"],
    }
)
riepilogo_problemi

## 5 · Refining a transformation

Let's take the most typical transformation for this file: converting the amount from the text `1.234,56` to the number `1234.56`.

We write it in two ways. First, the **«as it is»** version (*as-is* in English), meaning the approach that comes naturally to those coming from other languages: a loop that processes one row at a time. Then the **idiomatic** version, meaning code written in the way pandas is designed to be used: a single statement that works on the entire column.

### As-is: one row at a time

In [ ]:
# Work only on rows where the amount is in the correct format.
# copy() creates an independent table that we can modify without changing "grezzi".
validi = grezzi[importo_valido].copy()
validi["importo_numerico"] = 0.0              # prepare the column that will contain the result

inizio = time.perf_counter()                  # start the stopwatch

# iterrows returns the label and contents of each row, one at a time
for indice, riga in validi.iterrows():
    testo = riga["importo"]                   # for example "1.234,56"
    testo = testo.replace(".", "")            # remove the thousands separator: "1234,56"
    testo = testo.replace(",", ".")           # replace the decimal comma with a point: "1234.56"
    validi.loc[indice, "importo_numerico"] = float(testo)   # write the number into the corresponding row

tempo_ciclo = time.perf_counter() - inizio    # stop the stopwatch

print(f"Righe convertite: {len(validi)} - Tempo impiegato: {tempo_ciclo:.2f} secondi")

### Idiomatic: the entire column in a single statement

In [ ]:
inizio = time.perf_counter()

importo_numerico = (
    validi["importo"]
    .str.replace(".", "", regex=False)    # remove the thousands separator throughout the column
    .str.replace(",", ".", regex=False)   # replace the decimal comma with a point throughout the column
    .astype(float)                        # convert the entire column from text to numbers
)

tempo_colonna = time.perf_counter() - inizio

print(f"Righe convertite: {len(importo_numerico)} - Tempo impiegato: {tempo_colonna:.4f} secondi")
print(f"La versione idiomatica è circa {tempo_ciclo / tempo_colonna:.0f} volte più veloce.")
print("I due risultati coincidono:", importo_numerico.equals(validi["importo_numerico"]))

The two results are identical, but the idiomatic version is shorter, reads like a three-step recipe, and is hundreds of times faster. With twenty thousand rows, the difference is a few seconds; with twenty million rows, it is the difference between a pipeline that finishes in a few seconds and one that takes tens of minutes.

> **Rule of thumb.** When a pandas program contains a `for` loop that iterates over the rows of a table, there is almost always a statement that does the same work on the entire column.

## 6 · From the notebook to the project

The notebook is the right tool for **understanding**: we try things out, inspect the results, and make corrections. It is not the right tool for **putting a pipeline into production**, because the cells can be run in any order and the code is not easy to reuse or check automatically.

For this reason, once the rules were clear, we rewrote them in the project as **functions**, in the file `etl/trasformazione.py`. A function is a named piece of code that takes data as input and returns a result: it can be called from several places and checked with automated tests.

Let's call those functions from here, on the same file. Each returns two tables: the valid rows, which proceed to the next step, and the rejected records.

In [ ]:
from etl.estrazione import leggi_movimenti
from etl.trasformazione import controlla_formati, togli_duplicati

grezzi_dal_progetto = leggi_movimenti(FILE_MOVIMENTI)                 # read as text, including the row number
movimenti, scarti_formati = controlla_formati(grezzi_dal_progetto)    # validate and convert dates and amounts
movimenti, scarti_duplicati = togli_duplicati(movimenti)              # remove duplicate rows

print("Righe lette:              ", len(grezzi_dal_progetto))
print("Scartate per il formato:  ", len(scarti_formati))
print("Scartate perché ripetute: ", len(scarti_duplicati))
print("Righe che proseguono:     ", len(movimenti))

The counts match those found manually in section 4. Let's look at the column types after the transformation: the date has become an actual date (`datetime64`) and the amount an actual number (`float64`).

In [ ]:
movimenti.dtypes     # dtypes lists the data type of each column

In [ ]:
# For rejected records, keep the row number in the source file and the reason
scarti_formati.head()

At this point, it is worth moving on to the project files, in the order in which the pipeline uses them:

1. `esegui_etl.py`: the program to run;
2. `etl/flusso.py`: the steps of the pipeline, one after another;
3. `etl/trasformazione.py`: the rules covered in this notebook.

The pipeline is run from the VS Code terminal with the command:

```
python esegui_etl.py --data 2026-11-17
```

## 7 · After execution: querying the destination

This section must be run **after** running the pipeline at least once.

The destination is a **SQLite** database: a database contained in a single file, which does not require a server. Python handles it with the standard `sqlite3` module. With pandas, we can run a query written in **SQL**, the language of databases, and receive the result as a DataFrame.

In [ ]:
import sqlite3   # the standard Python module for SQLite databases

FILE_DATABASE = config["percorsi"]["file_database"]

# If the database does not exist yet, stop with a message explaining what to do
if not FILE_DATABASE.exists():
    raise FileNotFoundError(
        "Il database non esiste ancora. Eseguire prima il flusso dal terminale di VS Code:\n"
        "    python esegui_etl.py --data 2026-11-17"
    )

connessione = sqlite3.connect(FILE_DATABASE)   # the "connection" is the open channel to the database
print("Collegati a:", FILE_DATABASE)

### Execution history

The pipeline records every execution, successful or failed, in the `log_esecuzioni` table.

In [ ]:
storico = pd.read_sql_query(
    "SELECT * FROM log_esecuzioni ORDER BY id_esecuzione",
    connessione,
    dtype_backend="numpy_nullable",   # this keeps integers as integers even when a cell is empty
)
storico

### How many transactions were loaded, day by day

In [ ]:
pd.read_sql_query(
    '''
    SELECT data_elaborazione,
           COUNT(*)                   AS movimenti,
           ROUND(SUM(importo_eur), 2) AS saldo_eur,
           SUM(flag_rilevante)        AS movimenti_rilevanti
    FROM movimenti
    GROUP BY data_elaborazione
    ORDER BY data_elaborazione
    ''',
    connessione,
)

### The five branches with the most transactions for the day

The question mark in the query text is a **placeholder**: the value, in this case the date, is passed separately using `params`. This is the same technique the pipeline uses to write to the database.

In [ ]:
pd.read_sql_query(
    '''
    SELECT codice_filiale,
           SUM(numero_movimenti)     AS movimenti,
           ROUND(SUM(totale_eur), 2) AS saldo_eur
    FROM riepilogo_giornaliero
    WHERE data_elaborazione = ?
    GROUP BY codice_filiale
    ORDER BY movimenti DESC
    LIMIT 5
    ''',
    connessione,
    params=("2026-11-17",),
)

### Rejected rows

Rejected records do not go into the database but into a CSV file, one per day. Each row contains the row number in the source file, the reason for rejection, and the original content, written exactly as it was in the file.

In [ ]:
FILE_SCARTI = config["percorsi"]["cartella_scarti"] / "scarti_movimenti_20261117.csv"
scarti = pd.read_csv(FILE_SCARTI, sep=";", dtype=str, keep_default_na=False)

print(scarti["motivo_scarto"].value_counts())
scarti.head(10)

In [ ]:
connessione.close()   # close the database connection when we are done

## In summary

- Before building a pipeline, **look at the data**: the notebook and Data Wrangler are the tools for doing this.
- Read the files **as text**, check the format, and convert only what passes the check: automatic conversion can make mistakes without warning.
- Incorrect rows are not discarded or corrected arbitrarily: they become **rejected records**, each with its own reason.
- In pandas, work **by column**, not row by row: the code is shorter and much faster.
- The rules refined in the notebook become project **functions**, which can be reused and checked with tests.

The next step is the notebook `02_grandi_volumi.ipynb`, where we see what to do when there are several million rows rather than twenty thousand.